### RAG Pipelines- Data Ingestion to VectorDB pipeline

In [1]:
import os
from langchain_community.document_loaders import  PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\goura\AppData\Local\Temp\ipykernel_8132\3183071191.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import  PyPDFLoader, PyMuPDFLoader
d:\GOURAB DAS\CODE\Git\Agentic AI\Traditional RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory) :
    """Peocess all pdf files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all pdf files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files :
        print(f"\nprocessing: {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents :
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"\nloaded: {len(documents)}")

        except Exception as e:
            print(f"X Error: {e}")
        
    print(f"\nTotal documents loaded: {len(all_documents)}")

    return all_documents

# process all PDF's in the data directory
all_pdf_documents = process_all_pdfs("../data")



Found 3 PDF files to process

processing: git-cheat-sheet-education.pdf


Ignoring wrong pointing object 11 0 (offset 0)



loaded: 2

processing: Object Oriented Programming (1) (1).pdf

loaded: 16

processing: Operating System Notes.pdf

loaded: 12

Total documents loaded: 30


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Mac OS X 10.9.1 Quartz PDFContext', 'creator': 'Adobe Illustrator CC (Macintosh)', 'creationdate': "D:20140224195805Z00'00'", 'title': 'git-cheat-sheet-education', 'moddate': "D:20140224195805Z00'00'", 'source': '..\\data\\pdf\\git-cheat-sheet-education.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'git-cheat-sheet-education.pdf', 'file_type': 'pdf'}, page_content="GIT CHEAT SHEET\nSTAGE & SNAPSHOT\nWorking with snapshots and the Git staging area\ngit status\nshow modiﬁed ﬁles in working directory, staged for your next commit\ngit add [file]\nadd a ﬁle as it looks now to your next commit (stage)\ngit reset [file]\nunstage a ﬁle while retaining the changes in working directory\ngit diff\ndiﬀ of what is changed but not staged\ngit diff --staged\ndiﬀ of what is staged but not yet committed\ngit commit -m “[descriptive message]”\ncommit your staged content as a new commit snapshot\nSETUP\nConﬁguring user information used across all lo

In [4]:
# Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # show example of a chunk
    if split_docs :
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs



In [5]:
chunks = split_documents(all_pdf_documents)
chunks

Split 30 documents into 57 chunks

Example chunk:
Content: GIT CHEAT SHEET
STAGE & SNAPSHOT
Working with snapshots and the Git staging area
git status
show modiﬁed ﬁles in working directory, staged for your next commit
git add [file]
add a ﬁle as it looks now...
Metadata: {'producer': 'Mac OS X 10.9.1 Quartz PDFContext', 'creator': 'Adobe Illustrator CC (Macintosh)', 'creationdate': "D:20140224195805Z00'00'", 'title': 'git-cheat-sheet-education', 'moddate': "D:20140224195805Z00'00'", 'source': '..\\data\\pdf\\git-cheat-sheet-education.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'git-cheat-sheet-education.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Mac OS X 10.9.1 Quartz PDFContext', 'creator': 'Adobe Illustrator CC (Macintosh)', 'creationdate': "D:20140224195805Z00'00'", 'title': 'git-cheat-sheet-education', 'moddate': "D:20140224195805Z00'00'", 'source': '..\\data\\pdf\\git-cheat-sheet-education.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'git-cheat-sheet-education.pdf', 'file_type': 'pdf'}, page_content='GIT CHEAT SHEET\nSTAGE & SNAPSHOT\nWorking with snapshots and the Git staging area\ngit status\nshow modiﬁed ﬁles in working directory, staged for your next commit\ngit add [file]\nadd a ﬁle as it looks now to your next commit (stage)\ngit reset [file]\nunstage a ﬁle while retaining the changes in working directory\ngit diff\ndiﬀ of what is changed but not staged\ngit diff --staged\ndiﬀ of what is staged but not yet committed\ngit commit -m “[descriptive message]”\ncommit your staged content as a new commit snapshot\nSETUP\nConﬁguring user information used across all lo

### Embedding and VectorStoreDb

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}") # we can create a function to use it also
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embedding for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embedding with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

# initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1387.28it/s]


Model loaded successfully. Embedding dimension: 384


Vector Store

In [8]:
class VectorStore:
    """Manages document embeddings in a chromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_document", persist_directory: str = "../data/vector_store") :
        """
        Initialize the vector store

        Args:
            collection_name: Name of the chromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize chromaDB client and collection"""
        try:
            # create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of Langchain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings) :
            raise ValueError("Number of documents mst match number of embeddings")

        print(f"Adding {len(documents)} documents to the vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepear metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store = VectorStore()
vector_store

Vector store initialized. collection: pdf_document
Existing documents in collection: 114


In [9]:
chunks

[Document(metadata={'producer': 'Mac OS X 10.9.1 Quartz PDFContext', 'creator': 'Adobe Illustrator CC (Macintosh)', 'creationdate': "D:20140224195805Z00'00'", 'title': 'git-cheat-sheet-education', 'moddate': "D:20140224195805Z00'00'", 'source': '..\\data\\pdf\\git-cheat-sheet-education.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'git-cheat-sheet-education.pdf', 'file_type': 'pdf'}, page_content='GIT CHEAT SHEET\nSTAGE & SNAPSHOT\nWorking with snapshots and the Git staging area\ngit status\nshow modiﬁed ﬁles in working directory, staged for your next commit\ngit add [file]\nadd a ﬁle as it looks now to your next commit (stage)\ngit reset [file]\nunstage a ﬁle while retaining the changes in working directory\ngit diff\ndiﬀ of what is changed but not staged\ngit diff --staged\ndiﬀ of what is staged but not yet committed\ngit commit -m “[descriptive message]”\ncommit your staged content as a new commit snapshot\nSETUP\nConﬁguring user information used across all lo

In [10]:
# Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

# Generate the Embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# Store in the vector database
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 57 texts...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches: 100%|██████████| 2/2 [00:02<00:00,  1.40s/it]


Generated embeddings with shape: (57, 384)
Adding 57 documents to the vector store...
Successfully added 57 documents to vector store
Total documents in collection: 171


### Retriever Pipeline From VectorStore

In [21]:
class RAGRetriever:
    """Handels query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]] :
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
        
        Return: 
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top k: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1-distance 

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i+1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vector_store, embedding_manager)



In [12]:
rag_retriever

In [13]:
rag_retriever.retrieve("what is OS?")

Retrieving documents for query: 'what is OS?'
Top k: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.78it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_db91cf62_28',
  'content': 'Operating Systems\n● AnOperatingSystemcanbedefinedasaninterfacebetweenuserandhardware.Itis responsible for the execution of all the processes, Resource Allocation, CPUmanagement, File Management and many other tasks. The purpose of an operatingsystem is to provide an environment in which a user can execute programs in aconvenient and efficient manner.\n●\nTypes of Operating Systems:',
  'metadata': {'creator': 'PyPDF',
   'source_file': 'Operating System Notes.pdf',
   'source': '..\\data\\pdf\\Operating System Notes.pdf',
   'page_label': '1',
   'doc_index': 28,
   'page': 0,
   'content_length': 382,
   'title': 'Operating System Notes',
   'producer': 'Skia/PDF m93 Google Docs Renderer',
   'file_type': 'pdf',
   'total_pages': 12,
   'creationdate': ''},
  'similarity_score': 0.31996726989746094,
  'distance': 0.6800327301025391,
  'rank': 1},
 {'id': 'doc_c1abc621_28',
  'content': 'Operating Systems\n● AnOperatingSystemcanbedefinedasanint

### Integration Vectordb Context pipeline with LLM output

In [22]:
# Simple RAG pipeline with Groq LLM
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

# Initialize the Groq llm
model = "openai/gpt-oss-120b"
llm = ChatGroq(model= model)

# Simple RAG function: reteieve context + generate response
def rag_simple(query, retriever, llm, top_k: int=5) :
    # retriever the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    # generate the answer using GROQ LLM
    prompt = f"""
        Use the following context to answer the question concisely.
        context: {context}
        
        question: {query}
        Answer:
    """
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content


In [23]:
query = "What is operating system"
ans = rag_simple(query, rag_retriever, llm)

print(ans)

Retrieving documents for query: 'What is operating system'
Top k: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 15.93it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


An operating system is software that acts as an interface between users and computer hardware, managing resources (CPU, memory, I/O, files) and providing an environment in which programs can be executed efficiently and conveniently.


In [26]:
def rag_advance(query, retriever, llm, top_k: int = 5, min_score = 0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Return answer, sources, confidence score, and optionally full context.
    """

    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results :
        return {'ans': 'No relevant context found..', 'source': [], 'confidence': 0.0, 'context':''}
    
    # prepear context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300]+'...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""
        Use the following context to answer the question concisely.

        context: {context}
        question: {query}
        answer: 
    """
    response = llm.invoke([prompt.format(context, query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context :
        output['context'] = context
    return output


In [28]:
# Example usage:
results = rag_advance(query, rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", results['answer'])
print("Sources:", results['sources'])
print("Confidence:", results['confidence'])
print("Context Preview:", results['context'][:300])

Retrieving documents for query: 'What is operating system'
Top k: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.29it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: An operating system is the software layer that acts as an interface between users (and their programs) and the computer’s hardware, managing processes, CPU scheduling, memory and other resources, file handling, and providing a convenient, efficient environment for executing applications.
Sources: [{'source': 'Operating System Notes.pdf', 'page': 0, 'score': 0.5336046814918518, 'preview': 'Operating Systems\n● AnOperatingSystemcanbedefinedasaninterfacebetweenuserandhardware.Itis responsible for the execution of all the processes, Resource Allocation, CPUmanagement, File Management and many other tasks. The purpose of an operatingsystem is to provide an environment in which a user can e...'}, {'source': 'Operating System Notes.pdf', 'page': 0, 'score': 0.5336046814918518, 'preview': 'Operating Systems\n● AnOperatingSystemcanbedefinedasaninterfacebetweenuserandhardware.Itis responsible for the execution of all the processes, Resource Allocation, CPUmanagement, File Management and 